# An N-body star cluster

A Plummer sphere is a self-consistent equilibrium star cluster: its density $\rho \propto (1 + r^2/a^2)^{-5/2}$ is
held up by random stellar motions. `ps.scenarios.plummer_sphere` samples one; here 2000 stars ($G = M = a = 1$,
softening 0.02) are evolved with Barnes-Hut tree gravity for about four crossing times. In equilibrium the
**Lagrangian radii** (radii enclosing 10%, 50% and 90% of the mass) stay put and the virial ratio $2K/|U|$ stays near 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import physim as ps

In [ ]:
n, eps = 2000, 0.02
w = ps.scenarios.plummer_sphere(n, seed=1, softening=eps)
w.clear_forces()
w.add_force(ps.TreeGravity(G=1.0, softening=eps, theta=0.6))

# Tree accuracy against direct summation at this N.
direct = ps.scenarios.plummer_sphere(n, seed=1, softening=eps).accelerations()
err = np.linalg.norm(w.accelerations() - direct, axis=1) / np.linalg.norm(direct, axis=1)
print(f"tree force error at theta = 0.6: median {np.median(err):.1e}, 99th percentile {np.percentile(err, 99):.1e}")

e0 = w.total_energy()
tr = w.run(dt=0.01, steps=1000, record_every=20)
print(f"max |dE/E0| = {np.abs(ps.relative_energy_error(tr)).max():.1e}; "
      f"virial ratio 2K/|U| from {2 * tr.kinetic[0] / -tr.potential[0]:.3f} to {2 * tr.kinetic[-1] / -tr.potential[-1]:.3f}")

In [ ]:
com = np.einsum("fni,n->fi", tr.pos, w.masses) / w.masses.sum()
r = np.sort(np.linalg.norm(tr.pos - com[:, None], axis=2), axis=1)
fractions = [0.1, 0.5, 0.9]
# Plummer: the radius enclosing a mass fraction f is a / sqrt(f^(-2/3) - 1).
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
for k, f in enumerate(fractions):
    a1.plot(tr.t, r[:, int(f * n)], color=ps.plot.color(k), label=f"{f:.0%} of the mass")
    a1.axhline(1 / np.sqrt(f ** (-2 / 3) - 1), color=ps.plot.color(k), ls=":", lw=1)
a1.set_yscale("log"); a1.set_xlabel("t"); a1.set_ylabel("Lagrangian radius"); ps.plot.tidy(a1)
a1.legend(frameon=False, ncol=3, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0))
xy = tr.pos[-1]
a2.scatter(xy[:, 0], xy[:, 1], s=1, color=ps.plot.color(0), alpha=0.5)
a2.set_xlim(-4, 4); a2.set_ylim(-4, 4); a2.set_aspect("equal"); a2.set_xlabel("x"); a2.set_ylabel("y")
a2.set_title(f"t = {tr.t[-1]:.0f}"); ps.plot.tidy(a2);

The dotted lines are the analytic Plummer radii $a/\sqrt{f^{-2/3}-1}$; the sampled cluster matches them and does not
drift (the 90% radius sits slightly inside because the sample is truncated at $10a$). The small energy error comes from the tree approximation and the time step; a smaller `theta` or `dt` reduces it.